# Model Selection and Comparison

This notebook helps you compare model choices and decide when to use local, hosted, or GPU-served models.


## Selection criteria

When choosing a model, compare:

- quality
- latency
- cost
- tool use
- multimodal ability
- deployment simplicity


In [ ]:
models = [
    {'name': 'small-local', 'quality': 0.72, 'latency_ms': 220, 'cost': 0.001},
    {'name': 'hosted-strong', 'quality': 0.91, 'latency_ms': 680, 'cost': 0.015},
    {'name': 'gpu-vllm', 'quality': 0.88, 'latency_ms': 160, 'cost': 0.006},
]

models


## Heuristic

Use a small model for routing and summarization, a stronger model for complex reasoning, and a GPU-served model when you need throughput with better quality.


In [ ]:
def choose_model_for_task(task: str) -> str:
    lowered = task.lower()
    if any(word in lowered for word in ['vision', 'image', 'pdf', 'audio']):
        return 'multimodal or OCR-capable model'
    if any(word in lowered for word in ['routing', 'classification', 'triage']):
        return 'small local model'
    if any(word in lowered for word in ['reasoning', 'agent', 'planning']):
        return 'strong hosted model or gpu-served model'
    return 'balanced general model'

choose_model_for_task('Need high-quality agent planning and tool use')


## Comparison trick

Do not choose based on quality alone. The best model is the one that can meet the task with the lowest acceptable cost and latency.


In [ ]:
def score_model(model: dict) -> float:
    return model['quality'] - (model['latency_ms'] / 1000) - (model['cost'] * 10)

sorted(models, key=score_model, reverse=True)


## Tip

For real deployments, keep a clean abstraction so you can switch between local, hosted, and vLLM-backed endpoints without rewriting the app.
